<a href="https://colab.research.google.com/github/mehakgupta0406/marketing-ab-test-analysis/blob/main/Marketing_ab_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Business question
Does the conversion rate differ between users shown a marketing
advertisement and users shown a public service announcement (PSA)?
italicized text
## Groups
- Ad group: users shown the marketing advertisement.
- PSA group: users shown a public service announcement.

## Primary metric
Conversion rate = number of converted users / total users in each group.

## Hypotheses
- Null hypothesis (H₀): The groups have equal conversion rates.
- Alternative hypothesis (H₁): The groups have different conversion rates.

## Statistical testing plan
- Use a two-sided two-proportion z-test to compare conversion rates.
- Use a significance level of 5% (α = 0.05).
- Calculate a 95% confidence interval for the difference
  in conversion rates (ad minus PSA).
- Assess statistical evidence alongside effect size and
  experiment limitations before drawing a business conclusion.

In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("marketing_AB.csv")

In [3]:
df.shape

(588101, 7)

In [4]:
df.head()

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Unnamed: 0     588101 non-null  int64 
 1   user id        588101 non-null  int64 
 2   test group     588101 non-null  object
 3   converted      588101 non-null  bool  
 4   total ads      588101 non-null  int64 
 5   most ads day   588101 non-null  object
 6   most ads hour  588101 non-null  int64 
dtypes: bool(1), int64(4), object(2)
memory usage: 27.5+ MB


In [6]:
df.drop("Unnamed: 0",axis=1,inplace=True)

In [7]:
df.shape

(588101, 6)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   user id        588101 non-null  int64 
 1   test group     588101 non-null  object
 2   converted      588101 non-null  bool  
 3   total ads      588101 non-null  int64 
 4   most ads day   588101 non-null  object
 5   most ads hour  588101 non-null  int64 
dtypes: bool(1), int64(3), object(2)
memory usage: 23.0+ MB


In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df.drop_duplicates(inplace=True)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   user id        588101 non-null  int64 
 1   test group     588101 non-null  object
 2   converted      588101 non-null  bool  
 3   total ads      588101 non-null  int64 
 4   most ads day   588101 non-null  object
 5   most ads hour  588101 non-null  int64 
dtypes: bool(1), int64(3), object(2)
memory usage: 23.0+ MB


In [12]:
df["user id"].duplicated().sum()

np.int64(0)

In [13]:
df['test group'].unique()

array(['ad', 'psa'], dtype=object)

In [14]:
import sqlite3

In [15]:
conn = sqlite3.connect(":memory:")

In [16]:
df.to_sql("marketing_ab", conn, index=False, if_exists="replace")

588101

In [17]:
query="""
Select "test group" as test_group, Count(*) as no_of_users from marketing_ab group by "test group" """

In [18]:
pd.read_sql_query(query, conn)

,test_group,no_of_users
0,ad,564577
1,psa,23524


In [19]:
query="""
Select "test group" as test_group, Count(*) as no_of_users , SUM(converted) as conversion from marketing_ab group by "test group" """

In [20]:
pd.read_sql_query(query, conn)

,test_group,no_of_users,conversion
0,ad,564577,14423
1,psa,23524,420


In [21]:
query="""
Select "test group" as test_group, Count(*) as no_of_users , SUM(converted) as conversion ,100.0*(SUM(converted))/Count(*) as conversion_rate from marketing_ab group by "test group" """

In [22]:
pd.read_sql_query(query, conn)

,test_group,no_of_users,conversion,conversion_rate
0,ad,564577,14423,2.554656
1,psa,23524,420,1.785411


In [23]:
result=pd.read_sql_query(query, conn).set_index("test_group")

In [24]:
lift=(result.loc["ad","conversion_rate"]-result.loc["psa","conversion_rate"])

In [25]:
print(f"{lift:.2f} absolute lift")

0.77 absolute lift


In [26]:
relative_lift=lift/result.loc["psa","conversion_rate"]*100

In [27]:
print(f"{relative_lift:.2f}%")

43.09%
